# Deep Agent 고급 기능 (Human-in-the-Loop · Skills · Memory)

[400_Deep_Agent.ipynb](400_Deep_Agent.ipynb)에서 배운 Deep Agent의 **harness 고급 기능** 3가지를 실습합니다.

| 기능 | 파라미터 | 역할 |
|---|---|---|
| **Human-in-the-Loop** | `interrupt_on` | 특정 도구 실행 전 일시정지 → 사람이 승인/수정/거부 후 재개 |
| **Skills** | `skills` | SKILL.md로 전문 워크플로우 정의, 필요할 때만 전체 로드 |
| **Memory** | `memory` | AGENTS.md 파일 기반의 영속적 컨텍스트 (대화 간 유지) |

**참고**:
- [LangChain 공식 문서 - Human-in-the-Loop](https://docs.langchain.com/oss/python/deepagents/human-in-the-loop)
- [LangChain 공식 문서 - Skills](https://docs.langchain.com/oss/python/deepagents/skills)
- [LangChain 공식 문서 - Memory](https://docs.langchain.com/oss/python/deepagents/memory)

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

WORK_DIR = os.getcwd()

## 1. Human-in-the-Loop (interrupt_on)

파일 삭제, 이메일 발송처럼 **되돌릴 수 없는 도구**는 실행 전에 사람의 승인을 받아야 합니다.
`interrupt_on` 파라미터에 도구 이름을 지정하면, 에이전트가 그 도구를 호출하려는 순간 실행이 **일시정지**됩니다.

- 체크포인터가 **필수**입니다 (일시정지된 상태를 저장했다가 재개해야 하므로)
- 사람은 `approve`(승인) / `edit`(인자 수정 후 실행) / `reject`(거부) 중 하나로 응답합니다

In [2]:
import uuid
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

# 위험한 작업을 수행하는 도구 (스텁)
@tool
def delete_file(file_path: str) -> str:
    """지정한 파일을 삭제합니다. 되돌릴 수 없는 작업입니다."""
    return f"'{file_path}' 파일이 삭제되었습니다. (시뮬레이션)"


hitl_agent = create_deep_agent(
    model="openai:gpt-5-mini",
    tools=[delete_file],
    system_prompt=(
        "당신은 파일 관리 도우미입니다. "
        "사용자가 파일 삭제를 요청하면 추가 확인 질문 없이 delete_file 도구를 즉시 호출하세요. "
        "실행 승인 여부는 시스템(Human-in-the-Loop)이 처리합니다."
    ),
    checkpointer=InMemorySaver(),          # HITL에는 체크포인터가 필수
    interrupt_on={"delete_file": True},    # 이 도구 실행 전에 일시정지
)

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

C:\Users\trimu\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### 1.1 인터럽트 발생 확인

도구 호출 직전에 실행이 멈추고, 결과에 `__interrupt__` 키가 생깁니다.
그 안에 **어떤 도구를 어떤 인자로 실행하려는지**(`action_requests`)와
**허용된 결정 종류**(`review_configs`)가 담겨 있습니다.

In [3]:
result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "temp.txt 파일을 삭제해줘."}]},
    config,
)

# 인터럽트 발생 확인
if "__interrupt__" in result:
    interrupt_value = result["__interrupt__"][0].value

    print("실행이 일시정지되었습니다!\n")
    print("승인 대기 중인 작업 (action_requests):")
    for req in interrupt_value["action_requests"]:
        print(f"  - 도구: {req['name']}, 인자: {req['args']}")

    print("\n허용된 결정 (review_configs):")
    for cfg in interrupt_value["review_configs"]:
        print(f"  - {cfg['action_name']}: {cfg['allowed_decisions']}")
else:
    print("인터럽트가 발생하지 않았습니다.")

실행이 일시정지되었습니다!

승인 대기 중인 작업 (action_requests):
  - 도구: delete_file, 인자: {'file_path': '/temp.txt'}

허용된 결정 (review_configs):
  - delete_file: ['approve', 'edit', 'reject']


### 1.2 승인 후 재개

`Command(resume={"decisions": [...]})`로 결정을 전달하면 멈췄던 지점부터 이어서 실행됩니다.
결정은 `action_requests` 순서대로 하나씩 전달합니다.

| 결정 | 형식 | 동작 |
|---|---|---|
| 승인 | `{"type": "approve"}` | 원래 인자 그대로 도구 실행 |
| 수정 | `{"type": "edit", "edited_action": {"name": ..., "args": {...}}}` | 인자를 고쳐서 실행 |
| 거부 | `{"type": "reject", "message": "..."}` | 실행하지 않고 사유를 에이전트에게 전달 |

In [4]:
from langgraph.types import Command

# 사람이 "승인" 결정을 내렸다고 가정하고 재개
resumed = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config,  # 동일한 thread_id 사용
)

resumed["messages"][-1].pretty_print()

================================== Ai Message ==================================

[{'type': 'text', 'text': '요청하신 /temp.txt 파일을 삭제했습니다.', 'annotations': [], 'id': 'msg_07d4499819bc2008006a4a4e03f9ac819682d8dfc59e3fc7c5'}]


## 2. Skills (SKILL.md)

03장의 Skills Pattern(Progressive Disclosure)을 Deep Agent는 **파일 기반으로 기본 제공**합니다.

- 각 스킬은 `skills/<스킬이름>/SKILL.md` 파일로 정의 (frontmatter에 `name`, `description`)
- **3단계 로딩**: 시작 시 name/description만 시스템 프롬프트에 포함 → 작업과 관련되면 에이전트가 `read_file`로 본문 로드 → 본문이 참조하는 스크립트/문서는 필요할 때만 로드
- `skills` 파라미터에 스킬 디렉토리 경로를 전달 (백엔드 root 기준 상대 경로, 슬래시 사용)

> **Windows 참고** (400 본문과 동일): 내장 `read_file` 도구는 Windows 경로를 지원하지 않으므로,
> 스킬 본문을 읽을 때는 execute 도구로 python 한 줄 명령을 사용하도록 안내합니다.

In [5]:
# 스킬 정의 파일 생성: skills/sales-report/SKILL.md
skill_dir = os.path.join(WORK_DIR, "skills", "sales-report")
os.makedirs(skill_dir, exist_ok=True)

skill_md = """---
name: sales-report
description: 매출 리포트 작성 요청을 처리할 때 사용하는 스킬. 리포트 구조와 작성 규칙을 정의한다.
---

# sales-report

## 리포트 작성 규칙
1. 리포트는 반드시 [요약] → [상세 분석] → [권고 사항] 3개 섹션으로 구성한다
2. 모든 금액은 천 단위 콤마를 사용한다 (예: 1,250원)
3. 마지막 줄에 '-- 자동 생성 리포트'를 붙인다
"""

with open(os.path.join(skill_dir, "SKILL.md"), "w", encoding="utf-8") as f:
    f.write(skill_md)

print("SKILL.md 생성 완료:", os.path.join(skill_dir, "SKILL.md"))

SKILL.md 생성 완료: C:\Users\trimu\OneDrive\Agentic_AI_GPT_MCP\04_Deep_Agent\skills\sales-report\SKILL.md


In [6]:
from deepagents.backends import LocalShellBackend

import locale

# 서브프로세스 출력은 부모 프로세스(주피터 커널)의 기본 인코딩으로 디코딩되므로,
# 자식 python의 출력 인코딩을 부모와 일치시킵니다 (한글 Windows: cp949).
# ':replace' — 인코딩할 수 없는 문자는 오류 대신 대체 문자로 처리
SUBPROCESS_ENC = locale.getpreferredencoding(False)

backend = LocalShellBackend(
    root_dir=WORK_DIR,
    env={
        "PATH": os.environ.get("PATH", "/usr/bin:/bin"),  # execute 도구가 python 명령을 찾는 데 필요
        "PYTHONIOENCODING": f"{SUBPROCESS_ENC}:replace",
    },
)

# NOTE: Windows에서는 "name must match directory name" 경고가 출력될 수 있으나
#       경로 비교 방식 차이로 인한 것으로, 스킬은 정상 등록됩니다.
skills_agent = create_deep_agent(
    model="openai:gpt-5-mini",
    backend=backend,
    skills=["skills/"],   # 백엔드 root 기준 스킬 디렉토리
    checkpointer=InMemorySaver(),
)

config2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

# NOTE: read_file 도구는 Windows 경로를 지원하지 않으므로 (400 본문 참고),
#       스킬 본문을 읽을 때 execute 도구를 사용하도록 안내합니다.
result = skills_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "지금 사용 가능한 스킬 목록과 각 스킬의 설명을 알려줘. "
        "그리고 sales-report 스킬의 SKILL.md 본문을 읽어 리포트 작성 규칙을 요약해줘. "
        "파일을 읽을 때 read_file 도구는 사용하지 말고, 반드시 execute 도구로 "
        "python -c \"print(open('skills/sales-report/SKILL.md', encoding='utf-8').read())\" "
        "한 줄 명령을 사용해."
    )}]},
    config2,
)

for msg in result["messages"]:
    if hasattr(msg, "pretty_print"):
        msg.pretty_print()

Skill 'sales-report' in C:\Users\trimu\OneDrive\Agentic_AI_GPT_MCP\04_Deep_Agent\skills\sales-report/SKILL.md does not follow Agent Skills specification: name 'sales-report' must match directory name 'C:\Users\trimu\OneDrive\Agentic_AI_GPT_MCP\04_Deep_Agent\skills\sales-report'. Consider renaming for spec compliance.


================================ Human Message =================================

지금 사용 가능한 스킬 목록과 각 스킬의 설명을 알려줘. 그리고 sales-report 스킬의 SKILL.md 본문을 읽어 리포트 작성 규칙을 요약해줘. 파일을 읽을 때 read_file 도구는 사용하지 말고, 반드시 execute 도구로 python -c "print(open('skills/sales-report/SKILL.md', encoding='utf-8').read())" 한 줄 명령을 사용해.
================================== Ai Message ==================================

[{'id': 'rs_0a56513102bac4c4006a4a4e05a9008196bea0f5407b5f05be', 'summary': [], 'type': 'reasoning', 'content': []}, {'arguments': '{"command":"python -c \\"print(open(\'skills/sales-report/SKILL.md\', encoding=\'utf-8\').read())\\""}', 'call_id': 'call_890HY5KLBeLPEZVellyJX5Y0', 'name': 'execute', 'type': 'function_call', 'id': 'fc_0a56513102bac4c4006a4a4e0ecddc81969bdac5664d7c46e4', 'status': 'completed'}]
Tool Calls:
  execute (call_890HY5KLBeLPEZVellyJX5Y0)
 Call ID: call_890HY5KLBeLPEZVellyJX5Y0
  Args:
    command: python -c "print(open('skills/sales-report/SKILL.md', encoding='utf-8').read())"


## 3. Memory (AGENTS.md)

`memory` 파라미터에 파일 경로를 지정하면, 에이전트가 시작할 때 그 파일을 **시스템 프롬프트로 로드**하고,
대화 중 새로 알게 된 선호는 내장 `edit_file` 도구로 **직접 갱신**합니다.
파일로 저장되므로 **대화(스레드)가 바뀌어도 유지**됩니다 — 코딩 스타일, 선호도, 가이드라인 저장에 적합합니다.

In [7]:
# 메모리 파일 생성: AGENTS.md
agents_md = """# 에이전트 메모리

## 사용자 선호
- 답변은 항상 한국어 존댓말로 작성한다
- 답변 마지막에 '— 분석팀 드림' 서명을 붙인다
"""

with open(os.path.join(WORK_DIR, "AGENTS.md"), "w", encoding="utf-8") as f:
    f.write(agents_md)

memory_agent = create_deep_agent(
    model="openai:gpt-5-mini",
    backend=backend,
    memory=["AGENTS.md"],   # 시작 시 시스템 프롬프트로 로드되는 메모리 파일
    checkpointer=InMemorySaver(),
)

# 새 스레드에서 질문 — 메모리의 선호(서명)가 반영되는지 확인
config3 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "네가 기억하고 있는 내 선호를 알려줘."}]},
    config3,
)

result["messages"][-1].pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_0c6167ae2aa2cd74006a4a4e191c4c8190a9cb774541507ff3', 'summary': [], 'type': 'reasoning', 'content': []}, {'type': 'text', 'text': "지금 제 기억에 저장된 사용자 선호는 다음과 같습니다.\n\n1. 답변은 항상 한국어 존댓말로 작성할 것.  \n2. 답변 마지막에 '— 분석팀 드림' 서명을 붙일 것.\n\n현재는 위 두 가지가 저장되어 있으며, 추가하거나 수정하실 내용이 있으면 알려주시면 바로 반영하겠습니다.  \n\n— 분석팀 드림", 'annotations': [], 'id': 'msg_0c6167ae2aa2cd74006a4a4e1d1e008190bfbb62dffe64f4e6'}]


## 주요 포인트 정리

1. **Human-in-the-Loop**: `interrupt_on`으로 위험한 도구 실행 전 일시정지 → `__interrupt__` 확인 → `Command(resume=...)`로 승인/수정/거부 (체크포인터 필수)
2. **Skills**: SKILL.md 파일로 전문 워크플로우 정의 — 설명만 먼저 노출하고 본문은 필요할 때만 로드 (Progressive Disclosure)
3. **Memory**: AGENTS.md 파일 기반 영속 컨텍스트 — 시작 시 로드, `edit_file`로 갱신, 대화 간 유지
4. 세 기능 모두 `create_deep_agent()`의 **파라미터 하나**로 활성화되며, 별도 그래프 구성이 필요 없습니다

## 실습문제: 이메일 발송 승인 워크플로우

본문에서 배운 Human-in-the-Loop를 새로운 도구에 그대로 적용해 보세요.
이번에는 승인(approve)이 아니라 **거부(reject)** 를 해봅니다.

1. **도구 정의**: `send_email(to, subject)` 스텁 도구를 만드세요. (발송 확인 문자열 반환)
2. **에이전트 생성**: `interrupt_on={"send_email": True}`와 체크포인터를 지정해 에이전트를 만드세요.
3. **인터럽트 확인**: `"팀장님께 '주간 보고' 제목으로 메일을 보내줘"` 요청 후 `__interrupt__`의 `action_requests`를 출력하세요.
4. **거부 후 재개**: `{"type": "reject", "message": "..."}` 결정으로 재개하고, 에이전트가 거부 사유를 받고 어떻게 응답하는지 확인하세요.
